In [ ]:
from typing import Any
from typing import Dict

import gymnasium as gym
import os
import optuna
import numpy as np
import torch
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from sb3_contrib.ppo_mask import MaskablePPO
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, Batch

In [ ]:
class GCNFeatureExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space: gym.spaces.Dict, features_dim: int = 0) -> None:
        super().__init__(observation_space, features_dim)
        self.matrix_shape = observation_space['matrix'].shape
        self.edge_index_shape = observation_space['edge_index'].shape
        self.features_shape = observation_space['features'].shape

        if self.matrix_shape == None or self.edge_index_shape == None or self.features_shape == None:
            raise ValueError("Observation space must contain 'matrix', 'edge_index', and 'features'.")
        
        self.node_features_dim = self.features_shape[1]

        self.gcn1 = GCNConv(self.node_features_dim, 64)
        self.gcn2 = GCNConv(64, 32)
        self.gcn3 = GCNConv(32, 16)

        matrix_size = self.matrix_shape[0] * self.matrix_shape[1]
        self.matrix1 = torch.nn.Linear(matrix_size, 64)
        self.matrix2 = torch.nn.Linear(64, 32)
        self.matrix3 = torch.nn.Linear(32, 16)

        self.feature_combine = torch.nn.Sequential(
            torch.nn.Linear(32, features_dim),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.1),
            torch.nn.Linear(features_dim, features_dim)
        )
        
        self.dropout = torch.nn.Dropout(0.1)

    def forward(self, observations) -> torch.Tensor:
        batch_size = observations['matrix'].shape[0]
        device = observations['matrix'].device
        
        # 배치 내 각 그래프에 대해 Data 객체 생성
        data_list = []
        for i in range(batch_size):
            node_features = observations['features'][i].float()
            edge_index = observations['edge_index'][i].long()
            
            # 0이 아닌 노드만 필터링 (텐서 연산 사용)
            non_zero_mask = (node_features != 0).any(dim=1)
            filtered_node_features = node_features[non_zero_mask]
            
            # 자기 자신과 연결되지 않은 엣지만 필터링
            edge_mask = edge_index[:, 0] != edge_index[:,1]
            filtered_edge_index = edge_index[edge_mask]
            
            # 빈 그래프 처리
            if filtered_node_features.size(0) == 0:
                # 빈 그래프인 경우 더미 노드 하나 생성
                filtered_node_features = torch.zeros((1, node_features.size(1)), device=device)
            
            if filtered_edge_index.size(1) == 0:
                # 빈 그래프인 경우 더미 엣지 하나 생성 (자기 자신과 연결)
                filtered_edge_index = torch.zeros((2, 1), dtype=torch.long, device=device)
            
            data = Data(x=filtered_node_features, edge_index=filtered_edge_index)
            data_list.append(data)
        
        # PyTorch Geometric의 배치 처리
        batch = Batch.from_data_list(data_list)

        if batch.batch.dtype != torch.long:
            batch.batch = batch.batch.long()
        
        # GCN 통과
        x = self.gcn1(batch.x, batch.edge_index)
        x = torch.relu(x)
        x = self.dropout(x)

        x = self.gcn2(x, batch.edge_index)
        x = torch.relu(x)
        x = self.dropout(x)

        x = self.gcn3(x, batch.edge_index)
        x = torch.relu(x)

        # Global mean pooling 적용
        graph_features = global_mean_pool(x, batch.batch)  # (batch_size, 16)
        
        # Matrix 처리 (flatten)
        matrix_features = observations['matrix'].float().flatten(start_dim=1)  # (batch_size, matrix_size)
        matrix_features = self.matrix1(matrix_features)
        matrix_features = torch.relu(matrix_features)
        matrix_features = self.matrix2(matrix_features)
        matrix_features = torch.relu(matrix_features)
        matrix_features = self.matrix3(matrix_features)
        matrix_features = torch.relu(matrix_features)
        matrix_features = self.dropout(matrix_features)
        
        # 그래프 특성과 매트릭스 특성 결합
        combined_features = torch.cat([graph_features, matrix_features], dim=1)
        
        # 최종 변환
        output = self.feature_combine(combined_features)
        
        return output        

In [3]:
ENV_ID = "SabreSwapEnv"

gym.register(
    id=ENV_ID, entry_point="utils.sabre_env_wot_wgc:SabreSwapEnv"
)

N_TRIALS = 2000
N_STARTUP_TRIALS = 10
N_EVALUATIONS = 5
N_TIMESTEPS = int(1e6)
EVAL_FREQ = int(N_TIMESTEPS / N_EVALUATIONS)
N_EVAL_EPISODES = 5

DEFAULT_HYPERPARAMS = {
    "policy": "MultiInputPolicy",
    "env": make_vec_env(ENV_ID, n_envs=4, env_kwargs={"start_level": 1}),
    "policy_kwargs": dict(
        features_extractor_class=GCNFeatureExtractor,
        features_extractor_kwargs=dict(features_dim=128),
    ),
}

d:\lab\circuit_route\.venv\Lib\site-packages\gymnasium\envs\registration.py:734: UserWarning: WARN: The environment is being initialised with render_mode='rgb_array' that is not in the possible render_modes ([]).
  logger.warn(


In [4]:
def sample_ppo_params(trial: optuna.Trial) -> Dict[str, Any]:
    """Sampler for PPO hyperparameters."""
    gamma: float = 1.0 - trial.suggest_float("gamma", 0.0001, 0.1, log=True)
    max_grad_norm: float = trial.suggest_float("max_grad_norm", 0.3, 5.0, log=True)
    gae_lambda: float = 1.0 - trial.suggest_float("gae_lambda", 0.001, 0.1, log=True)
    n_steps: int = 2 ** trial.suggest_int("exponent_n_steps", 3, 11)
    learning_rate: float = trial.suggest_float("lr", 1e-5, 1, log=True)
    ent_coef: float = trial.suggest_float("ent_coef", 0.0000001, 0.1, log=True)
    vf_coef: float = trial.suggest_float("vf_coef", 0.000001, 1.0, log=True)

    # Display true values.
    trial.set_user_attr("gamma_", gamma)
    trial.set_user_attr("gae_lambda_", gae_lambda)
    trial.set_user_attr("n_steps", n_steps)


    return {
        "n_steps": n_steps,
        "gamma": gamma,
        "gae_lambda": gae_lambda,
        "learning_rate": learning_rate,
        "ent_coef": ent_coef,
        "max_grad_norm": max_grad_norm,
        "vf_coef": vf_coef,
    }

In [ ]:
from typing import Any


class CurriculumCallback(BaseCallback):
    """Optimized callback for curriculum learning."""

    def __init__(self, max_level: int = 1000, min_training_epi: int = 1000 ,success_threshold: float = 0.8, verbose: int = 0):
        super().__init__(verbose)
        self.max_level = max_level
        self.success_threshold = success_threshold
        self.min_training_epi = min_training_epi
        
        self.level = 1
        self.current_success_rate = 0.0

        # 로깅 주기 설정 (매 스텝이 아닌 주기적으로)
        self.log_freq = 100
        self.step_count = 0
        self.episode_count = 0
        self.truncated_or_success_count = 0


    def on_training_start(self, locals_: dict[str, Any], globals_: dict[str, Any]) -> None:
        super().on_training_start(locals_, globals_)
        self.training_env.env_method("set_level", level=self.level)

    def _on_step(self) -> bool:
        dones = self.locals.get("dones", [])
        
        # 에피소드 완료된 환경들만 처리
        if any(dones):
            for done in dones:
                if done:
                    self.episode_count += 1
            success_rate = np.mean(self.training_env.env_method("get_success_rate"))
            self.current_success_rate = success_rate
            
            # 레벨 업 체크
            if self.episode_count >= self.min_training_epi:
                if success_rate >= self.success_threshold:
                    self.episode_count = 0
                    self.level = min(self.level + 1, self.max_level)
                    self.training_env.env_method("set_level", level=self.level)
                    
                    if self.verbose > 0:
                        print(f"Level increased to {self.level} (success rate: {success_rate:.3f})")
        
        # 주기적으로만 상세 로깅
        self.step_count += 1
        if self.step_count % self.log_freq == 0:
            self.logger.record("success_rate", self.current_success_rate)
            self.logger.record("level", self.level)
            
            # 환경 상태는 덜 자주 로깅
            if self.step_count % (self.log_freq * 5) == 0:
                front_layer_len = self.training_env.env_method("front_layer_size")
                swap_candidate_len = self.training_env.env_method("swap_candidate_size")
                reset_failed = self.training_env.env_method("get_reset_failed")
                
                self.logger.record("front_layer_size/mean", np.mean(front_layer_len))
                self.logger.record("swap_candidate_size/mean", np.mean(swap_candidate_len))
                self.logger.record("reset_failed/mean", np.mean(reset_failed))
        
        return True


In [ ]:
class TrialEvalCallback(EvalCallback):
    """Callback used for evaluating and reporting a trial."""

    def __init__(
        self,
        eval_env: gym.Env,
        trial: optuna.Trial,
        n_eval_episodes: int = 5,
        eval_freq: int = 10000,
        deterministic: bool = True,
        verbose: int = 0,
    ):
        super().__init__(
            eval_env=eval_env,
            n_eval_episodes=n_eval_episodes,
            eval_freq=eval_freq,
            deterministic=deterministic,
            verbose=verbose,
        )
        self.trial = trial
        self.eval_idx = 0
        self.is_pruned = False

    def _on_step(self) -> bool:
        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:
            train_level = self.training_env.env_method("get_level")
            self.eval_env.env_method("set_level", level=train_level[0])
            super()._on_step()
            self.eval_idx += 1
            self.trial.report(self.last_mean_reward, self.eval_idx)

            # Prune trial if need.
            if self.trial.should_prune():
                self.is_pruned = True
                return False
        return True

In [ ]:
def objective(trial: optuna.Trial) -> float:
    kwargs = DEFAULT_HYPERPARAMS.copy()
    # Sample hyperparameters.
    kwargs.update(sample_ppo_params(trial))
    # Create the RL model.
    model = PPO(**kwargs, tensorboard_log="./optuna_sb3_wg_wot/",verbose=1, device='cuda')
    # Create env used for evaluation.
    eval_env = Monitor(gym.make(ENV_ID, start_level=1))
    # Create the callback that will periodically evaluate and report the performance.
    eval_callback = TrialEvalCallback(
        eval_env, trial, n_eval_episodes=N_EVAL_EPISODES, eval_freq=EVAL_FREQ, deterministic=True
    )
    nan_encountered = False
    try:
        print("start learning")
        model.learn(N_TIMESTEPS, callback=[eval_callback, CurriculumCallback(verbose=1, success_threshold=0.9)])
        model.save(f"./optuna_sb3_wg_wot/saves/rl_model_{trial.number}")
        print("Learning end")
    except AssertionError as e:
        # Sometimes, random hyperparams can generate NaN.
        print(e)
        nan_encountered = True
    finally:
        # Free memory.
        model.env.close()
        eval_env.close()

    # Tell the optimizer that the trial failed.
    if nan_encountered:
        return float("nan")

    if eval_callback.is_pruned:
        raise optuna.exceptions.TrialPruned()

    return eval_callback.last_mean_reward


sampler = TPESampler(n_startup_trials=N_STARTUP_TRIALS)
# Do not prune before 1/3 of the max budget is used.
pruner = MedianPruner(n_startup_trials=N_STARTUP_TRIALS, n_warmup_steps=N_EVALUATIONS // 3)

if not os.path.exists("./optuna_sb3_wg_wot/study.db"):
    study = optuna.create_study(study_name="sb3_wg_wot",sampler=sampler, pruner=pruner, direction="maximize", storage="sqlite:///./optuna_sb3_wg_wot/study.db")
else:
    print("Loading existing study...")
    study = optuna.load_study(study_name="sb3_wg_wot", storage="sqlite:///./optuna_sb3_wg_wot/study.db")
try:
    study.optimize(objective, n_trials=N_TRIALS, timeout= 60 * 60 * 12)  # 12 hours
except KeyboardInterrupt:
    
    pass

print("Number of finished trials: ", len(study.trials))

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

print("  User attrs:")
for key, value in trial.user_attrs.items():
    print("    {}: {}".format(key, value))

Loading existing study...
Using cuda device
start learning
Logging to ./optuna_sb3_wg_wot/PPO_2


d:\lab\circuit_route\.venv\Lib\site-packages\torch_geometric\utils\convert.py:278: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:257.)
  data_dict[key] = torch.as_tensor(value)
d:\lab\circuit_route\.venv\Lib\site-packages\gymnasium\utils\passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `reset()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")
d:\lab\circuit_route\.venv\Lib\site-packages\gymnasium\utils\passive_env_checker.py:158: UserWarning: WARN: The obs returned by the `step()` method is not within the observation space.
  logger.warn(f"{pre} is not within the observation space.")


-----------------------------------
| front_layer_size/    |          |
|    mean              | 1        |
| level                | 1        |
| reset_failed/        |          |
|    mean              | 1.5      |
| rollout/             |          |
|    ep_len_mean       | 43.3     |
|    ep_rew_mean       | -296     |
| success_rate         | 0.01     |
| swap_candidate_size/ |          |
|    mean              | 4.25     |
| time/                |          |
|    fps               | 345      |
|    iterations        | 1        |
|    time_elapsed      | 11       |
|    total_timesteps   | 4096     |
-----------------------------------
------------------------------------------
| front_layer_size/       |              |
|    mean                 | 1            |
| level                   | 1            |
| reset_failed/           |              |
|    mean                 | 1.5          |
| rollout/                |              |
|    ep_len_mean          | 35.4         |
|    ep_

[W 2025-07-08 17:53:44,926] Trial 1 failed with parameters: {'gamma': 0.014359243318772252, 'max_grad_norm': 0.6457112527196988, 'gae_lambda': 0.03079115200847968, 'exponent_n_steps': 10, 'lr': 1.659315782100875e-05, 'ent_coef': 2.7726273400001006e-07, 'vf_coef': 3.6736896015788795e-05} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "d:\lab\circuit_route\.venv\Lib\site-packages\optuna\study\_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\psw04\AppData\Local\Temp\ipykernel_12752\1291392437.py", line 16, in objective
    model.learn(N_TIMESTEPS, callback=[eval_callback, CurriculumCallback(verbose=1, success_threshold=0.9)])
  File "d:\lab\circuit_route\.venv\Lib\site-packages\stable_baselines3\ppo\ppo.py", line 311, in learn
    return super().learn(
           ^^^^^^^^^^^^^^
  File "d:\lab\circuit_route\.venv\Lib\site-packages\stable_baselines3\common\on_policy_alg

Number of finished trials:  2
Best trial:
  Value:  -476.16999999999996
  Params: 
    gamma: 0.036298644581231686
    max_grad_norm: 0.31705945567621574
    gae_lambda: 0.0037464096817459054
    exponent_n_steps: 6
    lr: 0.00031947390534093457
    ent_coef: 5.680166776769292e-06
    vf_coef: 0.00010296400784307887
  User attrs:
    gae_lambda_: 0.9962535903182541
    gamma_: 0.9637013554187683
    n_steps: 64
